# data processing
refactored to df

## imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path
import configparser
config = configparser.ConfigParser()
config.read_file(open('privateconfig'))
resdir = Path(config['Datafolder']['data'])
workdir = Path(config['Codefolder']['workspace'])
os.chdir(workdir)

In [3]:
# analysis
from scipy.io import loadmat
from sklearn.decomposition import FastICA
from sklearn.datasets import make_regression
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV, Lasso
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


In [4]:
# misc
import pickle
from collections import defaultdict

In [5]:
# task
from env_config import Config
from firefly_task import ffacc_real
# from monkey_functions import *
# from InverseFuncs import *
from stable_baselines3 import TD3
import torch


In [6]:
from neural_plot_ult import *
import time
tic=time.time()
import warnings
warnings.filterwarnings('ignore')

# Pre IRC

convert the mat data file (with neural data) into (states, actions, tasks) for IRC.


## prepare

In [9]:
# const
bin_size = 1  # how many bin of DT
# num_bins = 24  # how many bins to use. use 2.4 s and discard the long trials.
# monkey_height = 10
DT = 0.006  # DT for raw data
# reward_boundary = 65
areas = ['PPC', 'PFC', 'MST']
worldscale = 200


# m = 'm51'
folder = 'mat'
dens = [0.0001, 0.0005, 0.001,  0.005]

# locals().update({m: {}})
figure_path = resdir/'figures'
# datapaths = [i for i in Pa1th(resdir/'mat_ruiyi').glob(f'{m}*.mat')]
datapaths = [i for i in Path(resdir/folder).glob(f'*.mat')]
session=[int(a.stem[-2:]) for a in datapaths]
# datapaths


In [10]:
# load raw data
df = pd.DataFrame() 
for idx, datapath in enumerate(datapaths):
    if datapath.stem[-1].isalpha():
        continue
    try:
        print(datapath.stem)
        data = loadmat(datapath)
        # print(data.keys())
        
        
        trials_behv = data['trials_behv'][0]
        trials_units = data['units'][0]
        units_area = np.array([v[0] for v in trials_units['brain_area']])

        trial_behv =trials_behv[0]
        trial_ts = trial_behv['continuous']['ts'][0][0].reshape(-1)
        t_mask = (trial_ts > 0) & (
            ~np.isnan(trial_behv['continuous']['ymp'][0][0].reshape(-1)))
        t_mask &= trial_ts < trial_behv['events']['t_stop'][0][0].reshape(-1)
        if t_mask.sum() > 0:
            # remove the first data point to avoid downsample error
            t_mask[np.where(t_mask == True)[0][0]] = False

        # task varaibles from data
        mx = trial_behv['continuous']['xmp'][0][0][t_mask]
        my = trial_behv['continuous']['ymp'][0][0][t_mask]
        fx = trial_behv['continuous']['xfp'][0][0][t_mask]
        fy = trial_behv['continuous']['yfp'][0][0][t_mask]
        eye_hor_theta = trial_behv['continuous']['yre'][0][0][t_mask] # use yle for left eye.
        eye_ver_theta = trial_behv['continuous']['zre'][0][0][t_mask]
        # print(len(eye_hor_theta), len(eye_ver_theta), len(mx))
        mv = trial_behv['continuous']['v'][0][0][t_mask].reshape(-1, 1)
        mw = trial_behv['continuous']['w'][0][0][t_mask].reshape(-1, 1)

        print(len(trials_units),set(units_area))
        for trial_behv in trials_behv:
            a=(trial_behv['logical']['ptb'])[0][0][0][0]
            if a: print('has pert'); break


    except Exception as e:
        print(f"Error processing {datapath.stem}: {e}")
        continue


m44s209
76 {'MST', 'PPC'}
m44s221
73 {'MST', 'PPC'}
m53s116
53 {'MST', 'PPC', 'PFC'}
m44s220
76 {'MST', 'PPC'}
m44s208
94 {'MST', 'PPC'}
m44s183
105 {'MST', 'PPC'}
m53s128
97 {'PPC', 'PFC', 'VIP'}
m53s100
62 {'MST', 'PPC', 'PFC'}
m53s114
78 {'MST', 'PPC', 'PFC'}
m53s111
60 {'MST', 'PPC', 'PFC'}
m44s187
96 {'PPC', 'VIP'}
m44s218
87 {'MST', 'PPC'}
m53s39
137 {'PPC', 'PFC'}
has pert
m44s185
92 {'MST', 'PPC'}
m53s106
41 {'MST', 'PPC', 'PFC'}
m44s190
77 {'PPC', 'VIP'}
m44s219
77 {'MST', 'PPC'}
m53s48
123 {'PPC', 'PFC'}
has pert
m53s49
121 {'PPC', 'PFC'}
has pert
m53s98
54 {'MST', 'PPC', 'PFC'}
m53s95
50 {'PPC', 'PFC'}
m53s42
136 {'PPC', 'PFC'}
m53s43
131 {'PPC', 'PFC'}
m53s41
106 {'PPC', 'PFC'}
m53s40
121 {'PPC', 'PFC'}
has pert
m53s83
114 {'MST', 'PPC', 'PFC'}
m53s44
118 {'PPC', 'PFC'}
m53s50
148 {'PPC', 'PFC'}
has pert
m53s51
135 {'PPC', 'PFC'}
has pert
m53s92
54 {'MST', 'PPC', 'PFC'}
m53s86
80 {'MST', 'PPC', 'PFC'}
m53s90
44 {'MST', 'PPC', 'PFC'}
m53s47
104 {'PPC', 'PFC'}
m53s46
111 {'PP

In [ ]:

trials_behv = data['trials_behv'][0]
trials_units = data['units'][0]
units_area = np.array([v[0] for v in trials_units['brain_area']])

trial_behv =trials_behv[0]
trial_ts = trial_behv['continuous']['ts'][0][0].reshape(-1)
t_mask = (trial_ts > 0) & (
    ~np.isnan(trial_behv['continuous']['ymp'][0][0].reshape(-1)))
t_mask &= trial_ts < trial_behv['events']['t_stop'][0][0].reshape(-1)
if t_mask.sum() > 0:
    # remove the first data point to avoid downsample error
    t_mask[np.where(t_mask == True)[0][0]] = False

# task varaibles from data
mx = trial_behv['continuous']['xmp'][0][0][t_mask]
my = trial_behv['continuous']['ymp'][0][0][t_mask]
fx = trial_behv['continuous']['xfp'][0][0][t_mask]
fy = trial_behv['continuous']['yfp'][0][0][t_mask]
eye_hor_theta = trial_behv['continuous']['yre'][0][0][t_mask] # use yle for left eye.
eye_ver_theta = trial_behv['continuous']['zre'][0][0][t_mask]
# print(len(eye_hor_theta), len(eye_ver_theta), len(mx))
mv = trial_behv['continuous']['v'][0][0][t_mask].reshape(-1, 1)
mw = trial_behv['continuous']['w'][0][0][t_mask].reshape(-1, 1)

print(len(trials_units),set(units_area))
for trial_behv in trials_behv:
    a=(trial_behv['logical']['ptb'])[0][0][0][0]
    if a: print('has pert')


118 {'PPC', 'PFC'}


In [ ]:
trial_behv.dtype.names  

('continuous', 'events', 'logical', 'prs')

In [82]:
trial_behv['logical'].dtype.names  

('landmark_distance',
 'landmark_angle',
 'firefly_fullON',
 'replay',
 'landmark_fixedground',
 'reward',
 'ptb',
 'microstim',
 'spurioustarg')

In [91]:
for trial_behv in trials_behv:
    a=(trial_behv['logical']['ptb'])[0][0][0][0]
    if a: print(a)
    # print(a,len(a))


In [71]:
trial_behv['events'].dtype.names  

('t_beg',
 't_end',
 't_move',
 't_stop',
 't_sac',
 't_fix',
 't_rew',
 't_ptb',
 't_microstim',
 't_targ',
 't_beg_correction',
 't_flyON',
 't_flyON_minus_teleport')